# Train results — R2

**Strategy R2: protected repair (reject/rollback on regression)**

This notebook reproduces the `R2` column of the **Train results** tab in `Results.xlsx`, end to end. It:

1. Runs the repository's own analyzer (`scripts/analyze_repair_strategies.py`) on the `R2` training run (`experiments/subset_train-r2`).
2. Reads the analyzer's `overall_summary.csv` and `mechanism_summary.csv`.
3. Derives the **common-core** metrics (shared by every strategy) and the **R2-specific** metrics.
4. Writes the exact table to `Train_results_R2.xlsx`.

Everything is read-only analysis of an existing run: **no experiments, no model calls, no benchmark scoring.** The outputs below are produced by actually executing the cells.

## Step 0 — Setup and run the analyzer
We run from the repository root and point the analyzer at this strategy's training run. The analyzer writes its tables under a scratch output directory.

In [1]:
import subprocess, sys, os, csv
from pathlib import Path

# Resolve the repository root (the folder that contains 'scripts/' and 'experiments/').
ROOT = Path.cwd()
while not (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'analyze_repair_strategies.py').exists(), 'Run this from inside the repo.'
NB_DIR = Path.cwd()  # folder containing this notebook (before we chdir to repo root)
os.chdir(ROOT)
PY = sys.executable  # the kernel's interpreter (the project venv)

STRATEGY = 'R2'
RUN_DIR = 'experiments/subset_train-r2'
OUT_DIR = f'/tmp/train_nb_analysis/{STRATEGY}'

assert Path(RUN_DIR).is_dir(), f'Missing run folder: {RUN_DIR}'
print('Repo root :', ROOT)
print('Strategy  :', STRATEGY)
print('Run folder:', RUN_DIR)

Repo root : /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-
Strategy  : R2
Run folder: experiments/subset_train-r2


In [2]:
# Run the repo's analyzer for this one strategy. This only reads the saved run.
cmd = [PY, 'scripts/analyze_repair_strategies.py',
       '--run', f'{STRATEGY}={RUN_DIR}',
       '--output-dir', OUT_DIR, '--no-figures']
print('$', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout.strip()[-1500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])
assert res.returncode == 0, 'analyzer failed'
print('\nAnalyzer exit code:', res.returncode)

$ /Users/enruifung/.pyenv/versions/3.12.8/bin/python3 scripts/analyze_repair_strategies.py --run R2=experiments/subset_train-r2 --output-dir /tmp/train_nb_analysis/R2 --no-figures


R2: source=experiments/subset_train-r2, declared/evidenced queries=60, attempt queries=57, proposals=332, matrix cells=360

Analyzer exit code: 0


## Step 1 — Load the analyzer's summary tables
`overall_summary.csv` holds counts/means per run; `mechanism_summary.csv` holds the strategy-specific mechanism metrics.

In [3]:
def load_summary(path):
    d = {}
    with open(path) as f:
        for row in csv.DictReader(f):
            d[row['metric']] = row
    return d

overall = load_summary(f'{OUT_DIR}/{STRATEGY}/overall_summary.csv')
mech = load_summary(f'{OUT_DIR}/{STRATEGY}/mechanism_summary.csv')
print('overall_summary metrics:', len(overall))
print('mechanism_summary metrics:', len(mech))
print('\nmechanism metrics available:')
for k, v in mech.items():
    print(f'  {k}: count={v["count"]} valid_n={v["valid_n"]} rate={v["rate"]}')

overall_summary metrics: 85
mechanism_summary metrics: 5

mechanism metrics available:
  rejected: count=25 valid_n=332 rate=0.0753012048192771
  harmful_candidate_blocked: count=25 valid_n=332 rate=0.0753012048192771
  fixed_target_but_protected_regression_rejected: count=0 valid_n=332 rate=0
  rejected_and_continued: count=20 valid_n=332 rate=0.0602409638554217
  rejected_plan_differs_from_accepted: count=25 valid_n=25 rate=1


## Step 2 — Common-core metrics (shared by every strategy)
These come straight from `overall_summary.csv`. 'Confirmed unchanged attempts' is shown as `N (pct%)` of total attempts, matching the R1 template already in the spreadsheet.

In [4]:
def iv(metric):
    return int(float(overall[metric]['value']))

queries              = iv('declared_queries')
repair_attempts      = iv('total_attempts')
personal_attempts    = iv('personal_attempts')
commonsense_attempts = iv('commonsense_attempts')
unchanged            = iv('unchanged_confirmed')
unchanged_pct        = round(unchanged / repair_attempts * 100, 2)
personal_fixes       = iv('personal_fixed_count_sum')
personal_regressions = iv('personal_newly_violated_count_sum')

core = [
    ('Queries', queries),
    ('Repair attempts', repair_attempts),
    ('Personal repair attempts', personal_attempts),
    ('Commonsense repair attempts', commonsense_attempts),
    ('Confirmed unchanged attempts', f'{unchanged} ({unchanged_pct}%)'),
    ('Personal constraint fixes', personal_fixes),
    ('Personal regressions', personal_regressions),
]
for k, v in core:
    print(f'  {k:40} {v}')

  Queries                                  60
  Repair attempts                          332
  Personal repair attempts                 141
  Commonsense repair attempts              191
  Confirmed unchanged attempts             235 (70.78%)
  Personal constraint fixes                2
  Personal regressions                     0


## Step 3 — R2-specific metrics
These describe R2's own mechanism, read from `mechanism_summary.csv`. Each is formatted `count / valid_n (rate%)`.

In [5]:
SPECIFIC = [('Harmful candidates blocked (rejected)', 'harmful_candidate_blocked', 'count_rate'), ('Rejected: fixed target but caused a protected regression', 'fixed_target_but_protected_regression_rejected', 'count_rate'), ('Rejected and loop continued', 'rejected_and_continued', 'count_rate'), ('Rejected candidate differed from accepted plan', 'rejected_plan_differs_from_accepted', 'count_rate')]

def fmt_count_rate(row):
    c = int(float(row['count']))
    n = int(float(row['valid_n']))
    r = float(row['rate']) * 100
    return f'{c} / {n} ({r:.1f}%)'

specific = []
for label, metric, kind in SPECIFIC:
    if metric not in mech:
        specific.append((label, 'n/a (metric not produced for this run)'))
        continue
    row = mech[metric]
    specific.append((label, fmt_count_rate(row) if kind == 'count_rate' else row['rate']))
for k, v in specific:
    print(f'  {k:60} {v}')

  Harmful candidates blocked (rejected)                        25 / 332 (7.5%)
  Rejected: fixed target but caused a protected regression     0 / 332 (0.0%)
  Rejected and loop continued                                  20 / 332 (6.0%)
  Rejected candidate differed from accepted plan               25 / 25 (100.0%)


## Step 4 — Assemble the full table

In [6]:
rows = [('Metric', f'{STRATEGY} training result')]
rows += [(k, v) for k, v in core]
rows += [('', '')]  # spacer between core and strategy-specific
rows += [(k, v) for k, v in specific]

for k, v in rows:
    print(f'{k:60} | {v}')

Metric                                                       | R2 training result
Queries                                                      | 60
Repair attempts                                              | 332
Personal repair attempts                                     | 141
Commonsense repair attempts                                  | 191
Confirmed unchanged attempts                                 | 235 (70.78%)
Personal constraint fixes                                    | 2
Personal regressions                                         | 0
                                                             | 
Harmful candidates blocked (rejected)                        | 25 / 332 (7.5%)
Rejected: fixed target but caused a protected regression     | 0 / 332 (0.0%)
Rejected and loop continued                                  | 20 / 332 (6.0%)
Rejected candidate differed from accepted plan               | 25 / 25 (100.0%)


## Step 5 — Write the exact table to `Train_results_R2.xlsx`

In [7]:
import openpyxl
from openpyxl.styles import Font

wb = openpyxl.Workbook()
ws = wb.active
ws.title = f'{STRATEGY} train results'

for r, (k, v) in enumerate(rows, start=1):
    ws.cell(r, 1, k)
    ws.cell(r, 2, v)

# bold the header row
ws.cell(1, 1).font = Font(bold=True)
ws.cell(1, 2).font = Font(bold=True)
ws.column_dimensions['A'].width = 52
ws.column_dimensions['B'].width = 40

out_xlsx = NB_DIR / f'Train_results_{STRATEGY}.xlsx'
wb.save(out_xlsx)
print('Wrote', out_xlsx)
print('Rows written:', len(rows))

Wrote /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/train_result_analysis/Train_results_R2.xlsx
Rows written: 13


## Step 6 — Read the file back to confirm it saved correctly

In [8]:
chk = openpyxl.load_workbook(out_xlsx, data_only=True)
cs = chk.active
print('Sheet:', cs.title, '| dims:', cs.dimensions)
print()
for row in cs.iter_rows(values_only=True):
    if any(c not in (None, '') for c in row):
        print(f'{str(row[0]):52} | {row[1]}')

Sheet: R2 train results | dims: A1:B13

Metric                                               | R2 training result
Queries                                              | 60
Repair attempts                                      | 332
Personal repair attempts                             | 141
Commonsense repair attempts                          | 191
Confirmed unchanged attempts                         | 235 (70.78%)
Personal constraint fixes                            | 2
Personal regressions                                 | 0
Harmful candidates blocked (rejected)                | 25 / 332 (7.5%)
Rejected: fixed target but caused a protected regression | 0 / 332 (0.0%)
Rejected and loop continued                          | 20 / 332 (6.0%)
Rejected candidate differed from accepted plan       | 25 / 25 (100.0%)
